# 회원(members) CRUD

- 회원가입은 Supabase Auth(`auth.users`)로 하고, 추가 정보는 `members` 테이블에 저장
- RLS 때문에 **본인 정보만** 조회 / 수정 / 삭제 가능

In [1]:
from shop_client import new_client, login, TEST_PASSWORD

## Create: 회원가입 + members 등록

In [2]:
def sign_up_member(email, name):
    client = new_client()
    try:
        # 이미 가입한 계정이면 로그인
        client.auth.sign_in_with_password({"email": email, "password": TEST_PASSWORD})
    except Exception:
        client.auth.sign_up({"email": email, "password": TEST_PASSWORD})

    user = client.auth.get_user().user

    # members에 이미 있으면 건너뛰기
    exists = client.table("members").select("id").eq("id", user.id).execute()
    if not exists.data:
        client.table("members").insert({"id": user.id, "email": email, "name": name}).execute()

    print(f"가입 완료: {email} ({name})")
    return client


admin = sign_up_member("admin01@test.com", "관리자")
user1 = sign_up_member("user01@test.com", "김고객")
user2 = sign_up_member("user02@test.com", "이고객")
temp = sign_up_member("temp01@test.com", "탈퇴예정")

가입 완료: admin01@test.com (관리자)


가입 완료: user01@test.com (김고객)


가입 완료: user02@test.com (이고객)


가입 완료: temp01@test.com (탈퇴예정)


## Read: 조회 (RLS 때문에 본인 정보만 보임)

In [3]:
res = user1.table("members").select("email, name, role, phone").execute()
print("user01이 조회한 members:", res.data)

res = user2.table("members").select("email, name, role, phone").execute()
print("user02가 조회한 members:", res.data)

user01이 조회한 members: [{'email': 'user01@test.com', 'name': '김고객', 'role': 'customer', 'phone': None}]


user02가 조회한 members: [{'email': 'user02@test.com', 'name': '이고객', 'role': 'customer', 'phone': None}]


## Update: 이름, 전화번호 수정

In [4]:
user_id = user1.auth.get_user().user.id

print("수정 전:", user1.table("members").select("name, phone").eq("id", user_id).single().execute().data)

user1.table("members").update({"name": "김고객(수정)", "phone": "010-1234-5678"}).eq("id", user_id).execute()

print("수정 후:", user1.table("members").select("name, phone").eq("id", user_id).single().execute().data)

수정 전: {'name': '김고객', 'phone': None}


수정 후: {'name': '김고객(수정)', 'phone': '010-1234-5678'}


## 예외: 스스로 관리자(role)로 바꾸기 → 막힘

In [5]:
try:
    user1.table("members").update({"role": "admin"}).eq("id", user_id).execute()
except Exception as e:
    print("실패:", e)

print("role 그대로:", user1.table("members").select("role").eq("id", user_id).single().execute().data)

실패: {'message': 'permission denied for table members', 'code': '42501', 'hint': 'Grant the required privileges to the current role with: GRANT UPDATE ON public.members TO authenticated;', 'details': None}


role 그대로: {'role': 'customer'}


## 예외: 다른 사람 id로 members 등록 → RLS로 막힘

In [6]:
user2_id = user2.auth.get_user().user.id
try:
    user1.table("members").insert({"id": user2_id, "email": "fake@test.com", "name": "가짜"}).execute()
except Exception as e:
    print("실패:", e)

실패: {'message': 'new row violates row-level security policy for table "members"', 'code': '42501', 'hint': None, 'details': None}


## Delete: 회원 탈퇴 (본인 members 삭제)

In [7]:
temp_id = temp.auth.get_user().user.id

print("삭제 전:", temp.table("members").select("email").execute().data)
temp.table("members").delete().eq("id", temp_id).execute()
print("삭제 후:", temp.table("members").select("email").execute().data)

# 다른 사람 정보는 삭제해도 0건 (RLS)
res = user1.table("members").delete().eq("id", user2_id).execute()
print("user01이 user02 삭제 시도 → 삭제된 행:", len(res.data))

삭제 전: [{'email': 'temp01@test.com'}]


삭제 후: []
user01이 user02 삭제 시도 → 삭제된 행: 0
